In [2]:
import tensorflow as tf
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder

def prepare_data(df):
    """Simple preprocessing for Spaceship Titanic data"""
    # Create a copy to avoid modifying original data
    df = df.copy()
    
    # Define column types
    numeric_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    categorical_cols = ['HomePlanet', 'Destination']
    
    # Handle numeric columns
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')  # Convert to numeric
        df[col].fillna(df[col].mean(), inplace=True)
    
    # Handle categorical columns
    for col in categorical_cols:
        df[col].fillna('Unknown', inplace=True)
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
    
    # Scale numeric columns
    scaler = StandardScaler()
    df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
    
    # Convert to numpy array and ensure float32 dtype
    return df[numeric_cols + categorical_cols].astype('float32')

# Load your pre-split data
print("Loading data...")
train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')

# Save PassengerId for submission
test_ids = test_data['PassengerId']

# Prepare features
print("Preparing features...")
X_train = prepare_data(train_data)
X_test = prepare_data(test_data)

# Prepare target (ensure it's numeric)
y_train = train_data['Transported'].astype('float32')

print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")

# Create the model
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Compile
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train
print("Training model...")
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=3,
            restore_best_weights=True
        )
    ]
)

# Predict and create submission
print("Making predictions...")
predictions = model.predict(X_test)
predictions_binary = (predictions > 0.5).astype(bool)

# Create and save submission
submission_df = pd.DataFrame({
    'PassengerId': test_ids,
    'Transported': predictions_binary.flatten()
})
submission_df.to_csv('submission.csv', index=False)
print("Submission file created!")

# Print model summary
print("\nModel Summary:")
model.summary()

Loading data...
Preparing features...
Training data shape: (8693, 8)
Test data shape: (4277, 8)
Training model...
Epoch 1/20


/opt/miniconda3/envs/P1/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


218/218 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.6381 - loss: 0.6328 - val_accuracy: 0.7746 - val_loss: 0.4651
Epoch 2/20
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 977us/step - accuracy: 0.7524 - loss: 0.5035 - val_accuracy: 0.7786 - val_loss: 0.4659
Epoch 3/20
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 958us/step - accuracy: 0.7729 - loss: 0.4733 - val_accuracy: 0.7970 - val_loss: 0.4295
Epoch 4/20
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 964us/step - accuracy: 0.7688 - loss: 0.4748 - val_accuracy: 0.7970 - val_loss: 0.4241
Epoch 5/20
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 945us/step - accuracy: 0.7823 - loss: 0.4561 - val_accuracy: 0.7930 - val_loss: 0.4346
Epoch 6/20
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 945us/step - accuracy: 0.7873 - loss: 0.4560 - val_accuracy: 0.7976 - val_loss: 0.4179
Epoch 7/20
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7787 - loss: 0.4663 - val_accuracy: 0.7936 - val_loss: 0.4220
Epoch 8/20
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 951us/step - accuracy: 0.7784 - loss: 0.4674 - val_accuracy: 0

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,069 (31.52 KB)

 Trainable params: 2,689 (10.50 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 5,380 (21.02 KB)